# **Malaria Detection**

## Download Dataset

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

# Download latest version of the NIH/Kaggle Malaria dataset
dataset_download_path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("Path to dataset files:", dataset_download_path)


## 1. Import Required Libraries


In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (64, 64)
BATCH_SIZE = 32
CLASSES = ["Parasitized", "Uninfected"]  # index 0 = Parasitized, index 1 = Uninfected

## 2. Load Dataset

In [ ]:
DEFAULT_KAGGLE_PATH = "/kaggle/input/cell-images-for-detecting-malaria"
BASE_PATH = dataset_download_path if "dataset_download_path" in globals() and os.path.exists(dataset_download_path) else DEFAULT_KAGGLE_PATH


def find_dataset_root(base):
    """Walk down until we find the folder that directly contains both class dirs."""
    for dirpath, dirnames, _ in os.walk(base):
        if set(CLASSES).issubset(dirnames):
            return dirpath
    raise FileNotFoundError(f"Could not find {CLASSES} folders under {base}")


DATASET_PATH = find_dataset_root(BASE_PATH)
print(f"Loading data from: {DATASET_PATH}")

In [ ]:
rows = []
for label in CLASSES:
    for p in glob.glob(os.path.join(DATASET_PATH, label, "*.png")):
        rows.append({"filepath": p, "label": label})
df = pd.DataFrame(rows)
assert df["label"].nunique() == 2, "Expected 2 classes; check DATASET_PATH."

print("Class counts (full dataset):")
print(df["label"].value_counts())

## 3. Train / Validation / Test Split (70/15/15, stratified)

In [ ]:
trainval_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.15 / 0.85, stratify=trainval_df["label"], random_state=SEED
)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Augmentation ONLY for training; val/test get rescaling only.
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)


def make_gen(datagen, frame, shuffle):
    return datagen.flow_from_dataframe(
        frame,
        x_col="filepath",
        y_col="label",
        classes=CLASSES,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        shuffle=shuffle,
        seed=SEED,
    )


train_gen = make_gen(train_datagen, train_df, shuffle=True)
val_gen = make_gen(eval_datagen, val_df, shuffle=False)
test_gen = make_gen(eval_datagen, test_df, shuffle=False)

target_names = CLASSES
print("Class indices mapping:", test_gen.class_indices)

## 4. Model Architecture & Callbacks

In [ ]:
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu")(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D(2, 2)(x)

x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D(2, 2)(x)

x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D(2, 2)(x)

x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

malaria_detector = tf.keras.Model(inputs=inputs, outputs=outputs, name="malaria_detector")
malaria_detector.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# One monitored metric (val_loss) for all three callbacks, so they agree.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint("malaria_model.keras", monitor="val_loss", save_best_only=True, verbose=1),
]


## 5. Training & Test Evaluation

In [ ]:
history = malaria_detector.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks,
)

malaria_detector = tf.keras.models.load_model("malaria_model.keras")

# Test set was never used for early stopping, LR scheduling, or checkpointing.
loss, acc = malaria_detector.evaluate(test_gen, verbose=1)
print(f"\nKeras Test Accuracy: {acc * 100:.2f}%\n")

test_y = test_gen.classes
pred_probs = malaria_detector.predict(test_gen, verbose=1)
y_pred_classes = (pred_probs > 0.5).astype("int32").flatten()

print("Keras Classification Report (test set):")
print(classification_report(test_y, y_pred_classes, target_names=target_names))

cm = confusion_matrix(test_y, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Keras Model Confusion Matrix (Test Set)")
plt.show()


## 6. Grad-CAM

In [ ]:
last_conv_layer = next(
    l for l in reversed(malaria_detector.layers)
    if isinstance(l, tf.keras.layers.Conv2D)
)
grad_model = tf.keras.models.Model(
    inputs=malaria_detector.inputs,
    outputs=[last_conv_layer.output, malaria_detector.output],
)

sample_idx = 100
sample_img_path = test_gen.filepaths[sample_idx]
actual_label = target_names[test_gen.classes[sample_idx]]

img = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=IMG_SIZE)
img_arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0
img_tensor = tf.convert_to_tensor(np.expand_dims(img_arr, axis=0), dtype=tf.float32)

with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_tensor)
    p_uninfected = predictions[:, 0]
    predicted_idx = int(p_uninfected[0] > 0.5)
    target = p_uninfected if predicted_idx == 1 else 1.0 - p_uninfected

grads = tape.gradient(target, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

heatmap = tf.reduce_sum(pooled_grads * conv_outputs[0], axis=-1)
heatmap = np.maximum(heatmap.numpy(), 0)
if heatmap.max() != 0:
    heatmap /= heatmap.max()

heatmap = cv2.resize(heatmap, IMG_SIZE)
heatmap = np.uint8(255 * heatmap)
heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

orig_img = cv2.resize(cv2.imread(sample_img_path), IMG_SIZE)
superimposed_img = cv2.addWeighted(orig_img, 0.6, heatmap_color, 0.4, 0)

prob = float(p_uninfected[0])
predicted_label = target_names[predicted_idx]
confidence = prob * 100 if predicted_idx == 1 else (1 - prob) * 100

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
plt.title(f"Original: {actual_label}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.title(f"Grad-CAM: {predicted_label} ({confidence:.2f}%)")
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. TFLite : float baseline vs quantized

In [ ]:
float_converter = tf.lite.TFLiteConverter.from_keras_model(malaria_detector)
tflite_float_model = float_converter.convert()
float_path = "malaria_model_float.tflite"
with open(float_path, "wb") as f:
    f.write(tflite_float_model)

quant_converter = tf.lite.TFLiteConverter.from_keras_model(malaria_detector)
quant_converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = quant_converter.convert()
tflite_path = "malaria_model_quantized.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_quant_model)

size_keras = os.path.getsize("malaria_model.keras") / 1024
size_float = os.path.getsize(float_path) / 1024
size_quant = os.path.getsize(tflite_path) / 1024
print(f"\nKeras file (incl. optimizer state): {size_keras:.2f} KB")
print(f"TFLite float32: {size_float:.2f} KB | TFLite quantized: {size_quant:.2f} KB")
print(f"Quantization size reduction vs float TFLite: {(1 - size_quant / size_float) * 100:.2f}%\n")

## 8. Quantized Model Evaluation


In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

quant_y_true, quant_y_pred = [], []

print("Evaluating quantized model on the full test set...")
for batch_idx in range(len(test_gen)):
    x_batch, y_batch = test_gen[batch_idx]
    for x, y in zip(x_batch, y_batch):
        x_input = np.expand_dims(x, axis=0).astype(np.float32)
        interpreter.set_tensor(input_details[0]["index"], x_input)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        quant_y_pred.append(int(output.flatten()[0] > 0.5))
        quant_y_true.append(int(y))

quant_acc = accuracy_score(quant_y_true, quant_y_pred) * 100
print(f"\nQuantized Model Test Accuracy: {quant_acc:.2f}%\n")
print("Test-set class counts:", dict(pd.Series(quant_y_true).map(dict(enumerate(CLASSES))).value_counts()))
print("Quantized Model Classification Report:")
print(classification_report(quant_y_true, quant_y_pred, target_names=target_names))